In [3]:
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, cohen_kappa_score
import pickle
from pathlib import Path

VIT_LOGITS_PATH = (
    "/home/maria/ProjectionSort/data/"
    "google_vit-base-patch16-224_embeddings_logits.pkl"
)

def load_vit_array(vit_logits_path: str) -> np.ndarray:
    path = Path(vit_logits_path)
    if not path.exists():
        raise FileNotFoundError(f"ViT logits file not found: {path}")

    if path.suffix == ".npz":
        obj = np.load(path, allow_pickle=True)
        if "natural_scenes" not in obj:
            raise KeyError(f"Expected key 'natural_scenes'. Found keys: {list(obj.keys())}")
        vit = obj["natural_scenes"]
    elif path.suffix == ".npy":
        vit = np.load(path, allow_pickle=True)
    elif path.suffix in [".pkl", ".pickle"]:
        with open(path, "rb") as f:
            obj = pickle.load(f)
        if isinstance(obj, dict):
            if "natural_scenes" not in obj:
                raise KeyError(f"Expected key 'natural_scenes'. Found keys: {list(obj.keys())}")
            vit = obj["natural_scenes"]
        else:
            vit = obj
    else:
        raise ValueError(f"Unsupported ViT file extension: {path.suffix}")

    vit = np.asarray(vit)
    if vit.ndim != 2 or vit.shape[1] != 1000:
        raise ValueError(f"Expected ViT logits shape (n_images, 1000), got {vit.shape}")
    return vit

def load_vit_animate_labels(vit_logits_path: str) -> np.ndarray:
    vit = load_vit_array(vit_logits_path)
    top1 = np.argmax(vit, axis=1)
    image_labels = (top1 <= 397).astype(np.int64)

    print(f"Loaded ViT logits: {vit.shape}")
    print(f"Image-level label counts [inanimate, animate]: {np.bincount(image_labels)}")
    print("Convention: 0 = inanimate, 1 = animate")
    return image_labels

human = np.load("/home/maria/Science/data/image_labels.npy", allow_pickle=True).item()["labels"]
vit = load_vit_animate_labels(VIT_LOGITS_PATH)

mask = human != -1

human_labeled = human[mask]
vit_labeled = vit[mask]

print("Human counts [inanimate, animate]:", np.bincount(human_labeled))
print("ViT counts   [inanimate, animate]:", np.bincount(vit_labeled))

print("Accuracy:", accuracy_score(human_labeled, vit_labeled))
print("Cohen kappa:", cohen_kappa_score(human_labeled, vit_labeled))
print("Confusion matrix:")
print(confusion_matrix(human_labeled, vit_labeled))

Loaded ViT logits: (118, 1000)
Image-level label counts [inanimate, animate]: [55 63]
Convention: 0 = inanimate, 1 = animate
Human counts [inanimate, animate]: [62 56]
ViT counts   [inanimate, animate]: [55 63]
Accuracy: 0.940677966101695
Cohen kappa: 0.8817635270541082
Confusion matrix:
[[55  7]
 [ 0 56]]


In [6]:
import numpy as np
from scipy.stats import ttest_1samp
from statsmodels.stats.contingency_tables import mcnemar


human = np.load("/home/maria/Science/data/image_labels.npy", allow_pickle=True).item()["labels"]
vit = load_vit_animate_labels(VIT_LOGITS_PATH)

print("Human keys:", human)
print("ViT keys:", vit)

Loaded ViT logits: (118, 1000)
Image-level label counts [inanimate, animate]: [55 63]
Convention: 0 = inanimate, 1 = animate
Human keys: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 0 0 1 1 1 1 1
 0 0 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 1 0
 0 1 0 0 0 0 0]
ViT keys: [1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1 1 0 0 1 1 1 1 1
 0 0 1 1 1 1 0 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0
 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 0 1 0 1 1
 0 1 0 0 0 0 0]


In [7]:
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score, cohen_kappa_score
from statsmodels.stats.contingency_tables import mcnemar

human = np.load(
    "/home/maria/Science/data/image_labels.npy",
    allow_pickle=True
).item()["labels"]

vit = load_vit_animate_labels(VIT_LOGITS_PATH)

# Keep only manually labeled images
mask = human != -1

human = human[mask].astype(int)
vit = vit[mask].astype(int)

print("Human shape:", human.shape)
print("ViT shape:", vit.shape)

print("Human counts [inanimate, animate]:", np.bincount(human, minlength=2))
print("ViT counts   [inanimate, animate]:", np.bincount(vit, minlength=2))

print("Accuracy:", accuracy_score(human, vit))
print("Cohen kappa:", cohen_kappa_score(human, vit))

cm = confusion_matrix(human, vit, labels=[0, 1])

print()
print("Confusion matrix:")
print("rows = human labels [inanimate, animate]")
print("cols = ViT labels   [inanimate, animate]")
print(cm)

human_inanimate_vit_animate = np.where((human == 0) & (vit == 1))[0]
human_animate_vit_inanimate = np.where((human == 1) & (vit == 0))[0]

print()
print("Human inanimate, ViT animate:", human_inanimate_vit_animate)
print("Count:", len(human_inanimate_vit_animate))

print()
print("Human animate, ViT inanimate:", human_animate_vit_inanimate)
print("Count:", len(human_animate_vit_inanimate))

Loaded ViT logits: (118, 1000)
Image-level label counts [inanimate, animate]: [55 63]
Convention: 0 = inanimate, 1 = animate
Human shape: (118,)
ViT shape: (118,)
Human counts [inanimate, animate]: [62 56]
ViT counts   [inanimate, animate]: [55 63]
Accuracy: 0.940677966101695
Cohen kappa: 0.8817635270541082

Confusion matrix:
rows = human labels [inanimate, animate]
cols = ViT labels   [inanimate, animate]
[[55  7]
 [ 0 56]]

Human inanimate, ViT animate: [ 64  70  76  78  93 107 110]
Count: 7

Human animate, ViT inanimate: []
Count: 0


In [8]:
# rows = human label, cols = ViT label
table = confusion_matrix(human, vit, labels=[0, 1])

result = mcnemar(table, exact=True)

print("McNemar exact p-value:", result.pvalue)

McNemar exact p-value: 0.015625


In [9]:
import numpy as np

path = "/home/maria/Science/results/loo_pmc_vs_adam/loo_pmc_vs_adam_results.npz"
res = np.load(path, allow_pickle=True)

for k in res.files:
    arr = res[k]
    print(k, arr.shape, arr.dtype)

y (118,) int64
pmc_scores (118,) float64
pmc_preds (118,) int64
pmc_dirs (118, 39209) float32
adam_scores (118,) float64
adam_probs (118,) float64
adam_preds (118,) int64
adam_dirs (118, 39209) float32
adam_biases (118,) float64
pmc_stability (118, 118) float64
adam_stability (118, 118) float64
fold_cosines (118,) float64
fold_degrees (118,) float64


In [10]:
import numpy as np
import torch
from scipy.stats import ttest_1samp
from statsmodels.stats.contingency_tables import mcnemar
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix


# =============================================================================
# Config
# =============================================================================

NEURAL_PATH = "/home/maria/Science/data/hybrid_neural_responses_reduced.npy"
HUMAN_LABEL_PATH = "/home/maria/Science/data/image_labels.npy"

OUT_HUMAN = "/home/maria/Science/results/loo_pmc_vs_adam/human_adam_results.npz"
OUT_VIT = "/home/maria/Science/results/loo_pmc_vs_adam/vit_adam_results.npz"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

SEED = 0
N_EPOCHS = 1500
LR = 1e-2
WEIGHT_DECAY = 1e-3


# =============================================================================
# Utilities
# =============================================================================

def load_clean_neural(path):
    X = np.load(path)
    print("Raw neural shape:", X.shape)

    # You previously had raw shape (39209, 118), so transpose if needed.
    if X.shape[0] != 118 and X.shape[1] == 118:
        print("[INFO] Transposing neural matrix to images x features.")
        X = X.T

    X = X.astype(np.float32)

    # Remove bad columns.
    finite_cols = np.isfinite(X).all(axis=0)
    nonconstant_cols = X.std(axis=0) > 0
    good_cols = finite_cols & nonconstant_cols
    X = X[:, good_cols]

    print("Clean neural shape:", X.shape)
    print("Removed bad/nonconstant columns:", np.sum(~good_cols))

    return X


def standardize_train_test(X_train, X_test):
    mu = X_train.mean(axis=0, keepdims=True)
    sigma = X_train.std(axis=0, keepdims=True)
    sigma[sigma == 0] = 1.0

    X_train_z = (X_train - mu) / sigma
    X_test_z = (X_test - mu) / sigma

    return X_train_z, X_test_z


def fit_adam_logistic_one_fold(X_train, y_train, X_test):
    torch.manual_seed(SEED)

    Xtr = torch.tensor(X_train, dtype=torch.float32, device=DEVICE)
    ytr = torch.tensor(y_train, dtype=torch.float32, device=DEVICE)

    Xte = torch.tensor(X_test, dtype=torch.float32, device=DEVICE)

    n_features = Xtr.shape[1]

    w = torch.zeros(n_features, dtype=torch.float32, device=DEVICE, requires_grad=True)
    b = torch.zeros((), dtype=torch.float32, device=DEVICE, requires_grad=True)

    optimizer = torch.optim.Adam([w, b], lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = torch.nn.BCEWithLogitsLoss()

    for _ in range(N_EPOCHS):
        optimizer.zero_grad()
        logits = Xtr @ w + b
        loss = loss_fn(logits, ytr)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        test_logit = (Xte @ w + b).detach().cpu().numpy().ravel()[0]
        test_prob = 1.0 / (1.0 + np.exp(-test_logit))
        test_pred = int(test_prob >= 0.5)

        w_np = w.detach().cpu().numpy().copy()
        b_np = float(b.detach().cpu().numpy())

    return test_logit, test_prob, test_pred, w_np, b_np


def loo_adam_decoder(X, y, name="labels"):
    y = np.asarray(y).astype(np.int64)
    n = len(y)

    logits = np.zeros(n, dtype=np.float64)
    probs = np.zeros(n, dtype=np.float64)
    preds = np.zeros(n, dtype=np.int64)
    weights = []

    for i in range(n):
        train_idx = np.arange(n) != i
        test_idx = np.arange(n) == i

        X_train = X[train_idx]
        X_test = X[test_idx]
        y_train = y[train_idx]

        X_train_z, X_test_z = standardize_train_test(X_train, X_test)

        logit, prob, pred, w, b = fit_adam_logistic_one_fold(
            X_train_z,
            y_train,
            X_test_z,
        )

        logits[i] = logit
        probs[i] = prob
        preds[i] = pred
        weights.append(w)

        print(
            f"[{name} Adam LOO {i+1:03d}/{n}] "
            f"true={y[i]} logit={logit:+.6f} prob={prob:.4f} pred={pred}"
        )

    acc = accuracy_score(y, preds)
    bal_acc = balanced_accuracy_score(y, preds)
    auc = roc_auc_score(y, probs)
    cm = confusion_matrix(y, preds, labels=[0, 1])

    print()
    print("=" * 80)
    print(f"LOO Adam logistic axis: {name}")
    print("=" * 80)
    print(f"Accuracy:          {acc:.4f}")
    print(f"Balanced accuracy: {bal_acc:.4f}")
    print(f"AUC:               {auc:.4f}")
    print("Confusion matrix rows=true [inanimate, animate], cols=pred:")
    print(cm)

    return {
        "y_true": y,
        "adam_pred": preds,
        "adam_prob": probs,
        "adam_logit": logits,
        "adam_weights": np.stack(weights),
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "auc": auc,
        "confusion_matrix": cm,
    }


def save_results(path, result):
    np.savez_compressed(path, **result)
    print("Saved:", path)


def compare_human_vs_vit_adam(human_res, vit_res):
    human_y = human_res["y_true"]
    vit_y = vit_res["y_true"]

    human_pred = human_res["adam_pred"]
    vit_pred = vit_res["adam_pred"]

    human_correct = human_pred == human_y
    vit_correct = vit_pred == vit_y

    d = human_correct.astype(int) - vit_correct.astype(int)

    print()
    print("#" * 80)
    print("Human-label Adam vs ViT-label Adam")
    print("#" * 80)

    print("Human Adam accuracy:", human_correct.mean())
    print("ViT Adam accuracy:  ", vit_correct.mean())
    print("Difference:         ", d.mean())
    print("Extra correct images:", d.sum())

    print()
    print("Human-only correct:", np.sum((human_correct == 1) & (vit_correct == 0)))
    print("ViT-only correct:  ", np.sum((human_correct == 0) & (vit_correct == 1)))
    print("Both correct:      ", np.sum((human_correct == 1) & (vit_correct == 1)))
    print("Both wrong:        ", np.sum((human_correct == 0) & (vit_correct == 0)))

    # Paired t-test on image-level correctness difference.
    t_stat, p_two = ttest_1samp(d, popmean=0)
    p_one_human_greater = p_two / 2 if t_stat > 0 else 1 - p_two / 2

    print()
    print("Paired t-test on per-image correctness")
    print("t-stat:", t_stat)
    print("two-sided p:", p_two)
    print("one-sided p, human Adam > ViT Adam:", p_one_human_greater)

    # McNemar table:
    # rows = human wrong/correct
    # cols = vit wrong/correct
    table = np.array([
        [
            np.sum((human_correct == 0) & (vit_correct == 0)),
            np.sum((human_correct == 0) & (vit_correct == 1)),
        ],
        [
            np.sum((human_correct == 1) & (vit_correct == 0)),
            np.sum((human_correct == 1) & (vit_correct == 1)),
        ],
    ])

    print()
    print("McNemar table")
    print("rows = human Adam wrong/correct")
    print("cols = ViT Adam wrong/correct")
    print(table)

    mc = mcnemar(table, exact=True)
    print("McNemar exact p-value:", mc.pvalue)

    # Bootstrap CI for accuracy difference.
    rng = np.random.default_rng(SEED)
    B = 10_000
    boot = np.zeros(B)

    n = len(d)
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        boot[b] = d[idx].mean()

    lo, hi = np.percentile(boot, [2.5, 97.5])

    print()
    print("Bootstrap")
    print("Observed accuracy difference:", d.mean())
    print("95% bootstrap CI:", (lo, hi))
    print("Bootstrap p(diff <= 0):", np.mean(boot <= 0))

    return {
        "human_correct": human_correct,
        "vit_correct": vit_correct,
        "difference": d,
        "mcnemar_table": table,
        "mcnemar_p": mc.pvalue,
        "ttest_t": t_stat,
        "ttest_p_two_sided": p_two,
        "ttest_p_one_sided_human_greater": p_one_human_greater,
        "bootstrap_ci_95": (lo, hi),
        "bootstrap_p_diff_leq_0": np.mean(boot <= 0),
    }


# =============================================================================
# Run
# =============================================================================

X = load_clean_neural(NEURAL_PATH)

human = np.load(HUMAN_LABEL_PATH, allow_pickle=True).item()["labels"]
vit = load_vit_animate_labels(VIT_LOGITS_PATH)

mask = human != -1

X = X[mask]
human = human[mask].astype(np.int64)
vit = vit[mask].astype(np.int64)

print()
print("Human counts [inanimate, animate]:", np.bincount(human, minlength=2))
print("ViT counts   [inanimate, animate]:", np.bincount(vit, minlength=2))

human_res = loo_adam_decoder(X, human, name="human labels")
vit_res = loo_adam_decoder(X, vit, name="ViT labels")

save_results(OUT_HUMAN, human_res)
save_results(OUT_VIT, vit_res)

comparison = compare_human_vs_vit_adam(human_res, vit_res)

Raw neural shape: (39209, 118)
[INFO] Transposing neural matrix to images x features.
Clean neural shape: (118, 39209)
Removed bad/nonconstant columns: 0
Loaded ViT logits: (118, 1000)
Image-level label counts [inanimate, animate]: [55 63]
Convention: 0 = inanimate, 1 = animate

Human counts [inanimate, animate]: [62 56]
ViT counts   [inanimate, animate]: [55 63]


/home/maria/global_venv/lib/python3.12/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


[human labels Adam LOO 001/118] true=1 logit=-2.501769 prob=0.0757 pred=0


/home/maria/global_venv/lib/python3.12/site-packages/torch/cuda/__init__.py:789: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


[human labels Adam LOO 002/118] true=1 logit=-1.912057 prob=0.1287 pred=0
[human labels Adam LOO 003/118] true=1 logit=+8.257133 prob=0.9997 pred=1
[human labels Adam LOO 004/118] true=1 logit=+1.692082 prob=0.8445 pred=1
[human labels Adam LOO 005/118] true=1 logit=-4.209822 prob=0.0146 pred=0
[human labels Adam LOO 006/118] true=1 logit=+9.391195 prob=0.9999 pred=1
[human labels Adam LOO 007/118] true=1 logit=+4.381526 prob=0.9876 pred=1
[human labels Adam LOO 008/118] true=1 logit=+7.847923 prob=0.9996 pred=1
[human labels Adam LOO 009/118] true=1 logit=+10.904729 prob=1.0000 pred=1
[human labels Adam LOO 010/118] true=1 logit=+6.759705 prob=0.9988 pred=1
[human labels Adam LOO 011/118] true=1 logit=-1.428616 prob=0.1933 pred=0
[human labels Adam LOO 012/118] true=1 logit=-1.423850 prob=0.1941 pred=0
[human labels Adam LOO 013/118] true=1 logit=+1.512534 prob=0.8194 pred=1
[human labels Adam LOO 014/118] true=1 logit=+3.319336 prob=0.9651 pred=1
[human labels Adam LOO 015/118] true=

In [11]:
import numpy as np
from pathlib import Path
import pickle
import torch

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix
from scipy.stats import ttest_1samp
from statsmodels.stats.contingency_tables import mcnemar


# =============================================================================
# Paths
# =============================================================================

VIT_LOGITS_PATH = (
    "/home/maria/ProjectionSort/data/"
    "google_vit-base-patch16-224_embeddings_logits.pkl"
)

NEURAL_PATH = "/home/maria/Science/data/hybrid_neural_responses_reduced.npy"
HUMAN_LABEL_PATH = "/home/maria/Science/data/image_labels.npy"

OUTDIR = Path("/home/maria/Science/results/human_vs_vit_adam")
OUTDIR.mkdir(parents=True, exist_ok=True)


# =============================================================================
# Data loading
# =============================================================================

def load_vit_array(vit_logits_path: str) -> np.ndarray:
    path = Path(vit_logits_path)

    if not path.exists():
        raise FileNotFoundError(f"ViT logits file not found: {path}")

    if path.suffix == ".npz":
        obj = np.load(path, allow_pickle=True)
        vit = obj["natural_scenes"]

    elif path.suffix == ".npy":
        vit = np.load(path, allow_pickle=True)

    elif path.suffix in [".pkl", ".pickle"]:
        with open(path, "rb") as f:
            obj = pickle.load(f)

        if isinstance(obj, dict):
            vit = obj["natural_scenes"]
        else:
            vit = obj

    else:
        raise ValueError(f"Unsupported ViT file extension: {path.suffix}")

    vit = np.asarray(vit)

    if vit.ndim != 2 or vit.shape[1] != 1000:
        raise ValueError(f"Expected ViT logits shape (n_images, 1000), got {vit.shape}")

    return vit


def load_vit_animate_labels(vit_logits_path: str) -> np.ndarray:
    vit = load_vit_array(vit_logits_path)
    top1 = np.argmax(vit, axis=1)

    # ImageNet convention:
    # classes 0..397 are treated as animate.
    image_labels = (top1 <= 397).astype(np.int64)

    print(f"Loaded ViT logits: {vit.shape}")
    print(f"ViT label counts [inanimate, animate]: {np.bincount(image_labels, minlength=2)}")
    print("Convention: 0 = inanimate, 1 = animate")

    return image_labels


def load_human_labels(path: str) -> np.ndarray:
    labels = np.load(path, allow_pickle=True).item()["labels"]
    labels = np.asarray(labels).astype(np.int64)

    print(f"Loaded human labels: {labels.shape}")
    print(f"Human label counts [inanimate, animate], excluding -1:")
    print(np.bincount(labels[labels != -1], minlength=2))

    return labels


def load_clean_neural_and_labels(label_source: str):
    X = np.load(NEURAL_PATH, allow_pickle=True)
    X = np.asarray(X)

    if label_source == "human":
        y = load_human_labels(HUMAN_LABEL_PATH)
    elif label_source == "vit":
        y = load_vit_animate_labels(VIT_LOGITS_PATH)
    else:
        raise ValueError("label_source must be 'human' or 'vit'")

    print()
    print("=" * 80)
    print(f"Loading data for label source: {label_source}")
    print("=" * 80)
    print(f"Raw neural shape: {X.shape}")
    print(f"Labels shape:     {y.shape}")

    if X.shape[0] != len(y) and X.shape[1] == len(y):
        print("[INFO] Transposing neural matrix to images x features.")
        X = X.T

    if X.shape[0] != len(y):
        raise ValueError(f"Expected rows to match labels. Got X={X.shape}, y={y.shape}")

    # Remove unlabeled human images, if any.
    mask = y != -1
    X = X[mask]
    y = y[mask]

    finite_cols = np.all(np.isfinite(X), axis=0)
    nonconstant_cols = np.std(X[:, finite_cols], axis=0) > 1e-12

    good_cols = np.zeros(X.shape[1], dtype=bool)
    good_cols[np.where(finite_cols)[0][nonconstant_cols]] = True

    X = X[:, good_cols]

    print(f"Clean neural shape: {X.shape}")
    print(f"Removed bad/nonconstant columns: {np.sum(~good_cols)}")
    print(f"Final label counts [inanimate, animate]: {np.bincount(y, minlength=2)}")

    return X, y


# =============================================================================
# Adam logistic regression, matching your original script
# =============================================================================

def sigmoid_np(z):
    z = np.clip(z, -40, 40)
    return 1.0 / (1.0 + np.exp(-z))


class TorchLogisticRegression(torch.nn.Module):
    def __init__(self, n_features):
        super().__init__()
        self.linear = torch.nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x)


def fit_adam_logistic_axis(
    X_train,
    y_train,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=3000,
    seed=0,
):
    torch.manual_seed(seed)
    np.random.seed(seed)

    X_t = torch.tensor(X_train.astype(np.float32))
    y_t = torch.tensor(y_train.astype(np.float32)).view(-1, 1)

    model = TorchLogisticRegression(X_train.shape[1])

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
    )

    loss_fn = torch.nn.BCEWithLogitsLoss()

    for epoch in range(epochs):
        optimizer.zero_grad()
        logits = model(X_t)
        loss = loss_fn(logits, y_t)
        loss.backward()
        optimizer.step()

    with torch.no_grad():
        w = model.linear.weight.detach().cpu().numpy().ravel()
        b = float(model.linear.bias.detach().cpu().numpy()[0])

    return w, b


def loo_adam_scores(
    X,
    y,
    label_source: str,
    lr=1e-3,
    weight_decay=1e-4,
    epochs=3000,
):
    n, d = X.shape

    scores = np.zeros(n, dtype=np.float64)
    probs = np.zeros(n, dtype=np.float64)
    preds = np.zeros(n, dtype=np.int64)
    biases = np.zeros(n, dtype=np.float64)

    for test_idx in range(n):
        train_idx = np.arange(n) != test_idx

        X_train_raw = X[train_idx]
        y_train = y[train_idx]

        X_test_raw = X[~train_idx]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train_raw)
        X_test = scaler.transform(X_test_raw)

        # Important: this matches your original script.
        # Each LOO fold gets seed=test_idx.
        w, b = fit_adam_logistic_axis(
            X_train,
            y_train,
            lr=lr,
            weight_decay=weight_decay,
            epochs=epochs,
            seed=test_idx,
        )

        logit = float(X_test[0] @ w + b)
        prob = float(sigmoid_np(logit))
        pred = int(prob >= 0.5)

        scores[test_idx] = logit
        probs[test_idx] = prob
        preds[test_idx] = pred
        biases[test_idx] = b

        print(
            f"[{label_source} Adam LOO {test_idx + 1:03d}/{n}] "
            f"true={y[test_idx]} logit={logit:+.6f} prob={prob:.4f} pred={pred}"
        )

    return scores, probs, preds, biases


def summarize_adam(y, scores, probs, preds, label_source: str):
    acc = accuracy_score(y, preds)
    bal_acc = balanced_accuracy_score(y, preds)
    auc = roc_auc_score(y, probs)
    cm = confusion_matrix(y, preds, labels=[0, 1])

    print()
    print("=" * 80)
    print(f"LOO Adam logistic axis: {label_source} labels")
    print("=" * 80)
    print(f"Accuracy:          {acc:.4f}")
    print(f"Balanced accuracy: {bal_acc:.4f}")
    print(f"AUC:               {auc:.4f}")
    print("Confusion matrix rows=true [inanimate, animate], cols=pred:")
    print(cm)

    return {
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "auc": auc,
        "confusion_matrix": cm,
    }


def run_one_label_source(label_source: str):
    X, y = load_clean_neural_and_labels(label_source)

    print()
    print("#" * 80)
    print(f"Running LOO Adam logistic regression for {label_source} labels")
    print("#" * 80)

    adam_scores, adam_probs, adam_preds, adam_biases = loo_adam_scores(
        X,
        y,
        label_source=label_source,
        lr=1e-3,
        weight_decay=1e-4,
        epochs=3000,
    )

    metrics = summarize_adam(
        y,
        adam_scores,
        adam_probs,
        adam_preds,
        label_source=label_source,
    )

    outpath = OUTDIR / f"adam_{label_source}_labels_results.npz"

    np.savez(
        outpath,
        y=y,
        adam_scores=adam_scores,
        adam_probs=adam_probs,
        adam_preds=adam_preds,
        adam_biases=adam_biases,
        accuracy=metrics["accuracy"],
        balanced_accuracy=metrics["balanced_accuracy"],
        auc=metrics["auc"],
        confusion_matrix=metrics["confusion_matrix"],
    )

    print("Saved:", outpath)

    return {
        "label_source": label_source,
        "y": y,
        "adam_scores": adam_scores,
        "adam_probs": adam_probs,
        "adam_preds": adam_preds,
        "adam_biases": adam_biases,
        "metrics": metrics,
        "outpath": outpath,
    }


# =============================================================================
# Compare human-label Adam accuracy vs ViT-label Adam accuracy
# =============================================================================

def compare_human_vs_vit_adam(human_res, vit_res):
    human_y = human_res["y"]
    vit_y = vit_res["y"]

    human_preds = human_res["adam_preds"]
    vit_preds = vit_res["adam_preds"]

    human_correct = human_preds == human_y
    vit_correct = vit_preds == vit_y

    d = human_correct.astype(int) - vit_correct.astype(int)

    print()
    print("#" * 80)
    print("Human-label Adam vs ViT-label Adam")
    print("#" * 80)

    print(f"Human Adam accuracy: {human_correct.mean():.4f}")
    print(f"ViT Adam accuracy:   {vit_correct.mean():.4f}")
    print(f"Difference:          {d.mean():+.4f}")
    print(f"Extra correct images human - ViT: {d.sum()}")

    human_only = np.sum((human_correct == 1) & (vit_correct == 0))
    vit_only = np.sum((human_correct == 0) & (vit_correct == 1))
    both_correct = np.sum((human_correct == 1) & (vit_correct == 1))
    both_wrong = np.sum((human_correct == 0) & (vit_correct == 0))

    print()
    print(f"Human-only correct: {human_only}")
    print(f"ViT-only correct:   {vit_only}")
    print(f"Both correct:       {both_correct}")
    print(f"Both wrong:         {both_wrong}")

    # Paired t-test over image-level correctness differences.
    t_stat, p_two = ttest_1samp(d, popmean=0)
    p_one_human_greater = p_two / 2 if t_stat > 0 else 1 - p_two / 2

    print()
    print("Paired t-test on image-level correctness difference")
    print(f"t-stat: {t_stat:.6f}")
    print(f"two-sided p: {p_two:.6f}")
    print(f"one-sided p, human Adam > ViT Adam: {p_one_human_greater:.6f}")

    # McNemar table:
    # rows = human wrong/correct
    # cols = vit wrong/correct
    table = np.array([
        [
            np.sum((human_correct == 0) & (vit_correct == 0)),
            np.sum((human_correct == 0) & (vit_correct == 1)),
        ],
        [
            np.sum((human_correct == 1) & (vit_correct == 0)),
            np.sum((human_correct == 1) & (vit_correct == 1)),
        ],
    ])

    print()
    print("McNemar table")
    print("rows = human Adam wrong/correct")
    print("cols = ViT Adam wrong/correct")
    print(table)

    mc = mcnemar(table, exact=True)

    print(f"McNemar exact p-value: {mc.pvalue:.6f}")

    # Bootstrap CI for accuracy difference.
    rng = np.random.default_rng(0)
    B = 10_000
    boot = np.zeros(B)

    n = len(d)
    for b in range(B):
        idx = rng.integers(0, n, size=n)
        boot[b] = d[idx].mean()

    lo, hi = np.percentile(boot, [2.5, 97.5])

    print()
    print("Bootstrap")
    print(f"Observed accuracy difference: {d.mean():+.4f}")
    print(f"95% bootstrap CI: ({lo:+.4f}, {hi:+.4f})")
    print(f"Bootstrap p(diff <= 0): {np.mean(boot <= 0):.6f}")

    outpath = OUTDIR / "human_vs_vit_adam_comparison.npz"

    np.savez(
        outpath,
        human_correct=human_correct,
        vit_correct=vit_correct,
        correctness_difference=d,
        mcnemar_table=table,
        mcnemar_p=mc.pvalue,
        ttest_t=t_stat,
        ttest_p_two_sided=p_two,
        ttest_p_one_sided_human_greater=p_one_human_greater,
        bootstrap_ci_95=np.array([lo, hi]),
        bootstrap_p_diff_leq_0=np.mean(boot <= 0),
    )

    print("Saved comparison:", outpath)


# =============================================================================
# Main
# =============================================================================

def main():
    human_res = run_one_label_source("human")
    vit_res = run_one_label_source("vit")

    compare_human_vs_vit_adam(human_res, vit_res)


if __name__ == "__main__":
    main()

Loaded human labels: (118,)
Human label counts [inanimate, animate], excluding -1:
[62 56]

Loading data for label source: human
Raw neural shape: (39209, 118)
Labels shape:     (118,)
[INFO] Transposing neural matrix to images x features.
Clean neural shape: (118, 39209)
Removed bad/nonconstant columns: 0
Final label counts [inanimate, animate]: [62 56]

################################################################################
Running LOO Adam logistic regression for human labels
################################################################################
[human Adam LOO 001/118] true=1 logit=-2.554781 prob=0.0721 pred=0
[human Adam LOO 002/118] true=1 logit=-1.164098 prob=0.2379 pred=0
[human Adam LOO 003/118] true=1 logit=+12.931960 prob=1.0000 pred=1
[human Adam LOO 004/118] true=1 logit=+2.839264 prob=0.9448 pred=1
[human Adam LOO 005/118] true=1 logit=-5.751934 prob=0.0032 pred=0
[human Adam LOO 006/118] true=1 logit=+16.565349 prob=1.0000 pred=1
[human Adam LOO 007/11